In [ ]:
#!pip install -q -U langchain langchain-openai langchain-core langgraph python-dotenv

# Hooks en LangChain: pre y post hooks de agente y de modelo


Referencias:
- https://docs.langchain.com/oss/python/langchain/middleware/custom
- https://docs.langchain.com/oss/deepagents/code/hooks

Objetivos:

- Entender qué es un hook y los tipos que existen.
- Implementar **pre-hooks** y **post-hooks** a nivel de **agente** y de **modelo**.
- Ver el orden real de ejecución dentro del ciclo del agente.
- Compartir datos entre hooks con un estado propio y cortar la ejecución con `jump_to`.
- Conocer la forma de clase (`AgentMiddleware`), los hooks *wrap* y el orden con varios middleware.

> Aquí los hooks solo observan y controlan el flujo de forma básica. No son un esquema de seguridad: los *guardrails* se ven en la siguiente clase.

## Teoría: ¿qué es un hook?

Un agente de LangChain funciona como un ciclo: el modelo responde, si pide tools se ejecutan, y el modelo vuelve a responder hasta dar la respuesta final. Un **hook** es una función tuya que LangChain ejecuta automáticamente en un punto fijo de ese ciclo, sin modificar el prompt ni las tools. Se registran con el parámetro `middleware` de `create_agent`.

Para qué sirven: logs y métricas, contar tokens, limitar pasos, modificar el estado, o validar entradas y salidas (esto último lo profundizamos en la siguiente clase).

### Tipos de hooks

| Hook | Nivel | Cuándo corre |
|------|-------|--------------|
| `before_agent` (pre-hook de agente) | Agente | **1 vez** al inicio de cada invocación |
| `before_model` (pre-hook de modelo) | Modelo | **Antes de cada** llamada al LLM |
| `after_model` (post-hook de modelo) | Modelo | **Después de cada** respuesta del LLM |
| `after_agent` (post-hook de agente) | Agente | **1 vez** al terminar la invocación |
| `wrap_model_call` | Modelo | **Envuelve** cada llamada al LLM (decide si llamar al modelo) |
| `wrap_tool_call` | Tool | **Envuelve** cada ejecución de una tool |

Los cuatro primeros son *node-style*: corren en un punto y devuelven un diccionario que se mezcla con el estado (o `None`). Los dos últimos son *wrap-style*: reciben un `handler` y tú decides si lo llamas, cuántas veces y qué haces antes y después.

### Ciclo de ejecución

```
before_agent ─▶ [ before_model ─▶ LLM ─▶ after_model ─▶ tools ]* ─▶ after_agent
                 └───────────── se repite mientras el modelo pida tools ─────┘
```

Por eso, con una pregunta que usa una tool, los hooks de **modelo** corren 2 veces (una por cada llamada al LLM) y los de **agente** solo 1.

> Nota: Deep Agents y Claude Code también tienen hooks (`PreToolUse`, `PostToolUse`, `Stop`, etc.), pero son programas externos que reciben un JSON por stdin. La idea es la misma (observar y controlar el ciclo); lo que cambia es que los de LangChain son funciones Python dentro del mismo proceso.

In [ ]:
import getpass
import os

from dotenv import load_dotenv

# Carga el .env que está a la misma altura que este notebook
load_dotenv(".env")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # cambia el modelo si lo necesitas

## 1. Preparación: una tool y un agente sin hooks

Usamos una sola tool, `get_weather`, para provocar el ciclo completo modelo → tool → modelo. Primero un agente **sin hooks**, como punto de comparación.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def get_weather(location: str) -> str:
    """Devuelve el pronóstico del clima actual para la ubicación indicada."""
    return f"El clima en {location} es soleado con una temperatura de 22°C."


llm = ChatOpenAI(model=OPENAI_MODEL)

SYSTEM_PROMPT = "Eres un asistente útil y amigable. Responde siempre en español y sé conciso."

agent_sin_hooks = create_agent(model=llm, tools=[get_weather], system_prompt=SYSTEM_PROMPT)

response = agent_sin_hooks.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print(response["messages"][-1].content)

Sin hooks no vemos nada de lo que pasó adentro. Vamos a agregar un hook de cada tipo.

## 2. Pre-hook de agente: `@before_agent`

Corre **una vez** al inicio de la invocación. Recibe el `state` (con la lista `messages`) y el `runtime`. Aquí mostramos la pregunta del usuario. Devolvemos `None` porque no queremos cambiar el estado.

In [ ]:
from typing import Any

from langchain.agents.middleware import before_agent
from langgraph.runtime import Runtime


@before_agent
def pre_agent_hook(state, runtime: Runtime) -> dict[str, Any] | None:
    pregunta = state["messages"][-1].content
    print(f"[pre_agent] pregunta: {pregunta!r}")
    return None


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[pre_agent_hook],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print(response["messages"][-1].content)

## 3. Pre-hook de modelo: `@before_model`

Corre **antes de cada llamada al LLM**. Mostramos cuántos mensajes se le van a enviar. Fíjate que la pregunta usa una tool, así que el hook se ejecuta **dos veces**: la segunda vez el modelo ya recibe también el `AIMessage` con la tool y su `ToolMessage`.

In [ ]:
from langchain.agents.middleware import before_model


@before_model
def pre_model_hook(state, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[pre_model] llamando al modelo con {len(state['messages'])} mensaje(s)")
    return None


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[pre_model_hook],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print(response["messages"][-1].content)

## 4. Post-hook de modelo: `@after_model`

Corre **después de cada respuesta del LLM**. El último mensaje del estado es la respuesta del modelo (`AIMessage`), de donde sacamos las tools que pidió y los tokens consumidos.

In [ ]:
from langchain.agents.middleware import after_model


@after_model
def post_model_hook(state, runtime: Runtime) -> dict[str, Any] | None:
    ultimo = state["messages"][-1]
    tools_pedidas = [call["name"] for call in ultimo.tool_calls]
    uso = ultimo.usage_metadata or {}
    print(
        f"[post_model] tokens entrada={uso.get('input_tokens', 0)} "
        f"salida={uso.get('output_tokens', 0)}, tools pedidas={tools_pedidas or 'ninguna'}"
    )
    return None


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[post_model_hook],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print(response["messages"][-1].content)

## 5. Post-hook de agente: `@after_agent`

Corre **una vez** al terminar la invocación, con el estado final. Es el lugar natural para un resumen: aquí mostramos cuántos mensajes se generaron y el largo de la respuesta final.

In [ ]:
from langchain.agents.middleware import after_agent


@after_agent
def post_agent_hook(state, runtime: Runtime) -> dict[str, Any] | None:
    respuesta = state["messages"][-1].content
    print(f"[post_agent] {len(state['messages'])} mensajes en total, respuesta de {len(respuesta)} caracteres")
    return None


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[post_agent_hook],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print(response["messages"][-1].content)

## 6. Los cuatro hooks juntos

Registramos los cuatro en el mismo agente. La salida muestra el orden real de ejecución, que coincide con el diagrama de la teoría:

`pre_agent` → `pre_model` → `post_model` → *(tool)* → `pre_model` → `post_model` → `post_agent`

In [ ]:
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[pre_agent_hook, pre_model_hook, post_model_hook, post_agent_hook],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print("\n🤖", response["messages"][-1].content)

Con una pregunta que **no** usa tools, los hooks de modelo corren una sola vez:

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="Dime un saludo corto.")]})
print("\n🤖", response["messages"][-1].content)

## 7. Estado propio y `jump_to`

Los hooks pueden **compartir datos** a través del estado del agente. Para eso extendemos `AgentState` con campos propios (`NotRequired`, porque no existen al inicio) y pasamos el esquema con `state_schema`. Cuando un hook devuelve un diccionario, sus claves se mezclan con el estado.

Además un hook puede **cortar la ejecución** devolviendo `jump_to: "end"`. Para que LangChain lo permita se declara con `can_jump_to=["end"]`.

En este ejemplo:
- `pre_agent` inicializa el contador `model_calls` y la lista `tools_called`.
- `pre_model` incrementa el contador y, si pasa el límite, responde con un mensaje y termina.
- `post_model` acumula las tools que pide el modelo.
- `post_agent` muestra el resumen.

Primero con un límite holgado (5 llamadas):

In [ ]:
import time

from langchain.agents.middleware import AgentState
from langchain_core.messages import AIMessage
from typing_extensions import NotRequired


class HooksState(AgentState):
    started_at: NotRequired[float]
    model_calls: NotRequired[int]
    tools_called: NotRequired[list[str]]


def crear_hooks(max_model_calls: int):
    """Crea los 4 hooks con un límite de llamadas al modelo configurable."""

    @before_agent(state_schema=HooksState)
    def pre_agent(state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[pre_agent] pregunta: {state['messages'][-1].content!r}")
        return {"started_at": time.time(), "model_calls": 0, "tools_called": []}

    @before_model(state_schema=HooksState, can_jump_to=["end"])
    def pre_model(state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        llamadas = state.get("model_calls", 0) + 1
        print(f"[pre_model] llamada #{llamadas} al modelo")
        if llamadas > max_model_calls:
            print(f"[pre_model] límite de {max_model_calls} llamada(s) alcanzado → jump_to end")
            return {
                "messages": [AIMessage("No pude completar la solicitud: se alcanzó el límite de pasos.")],
                "jump_to": "end",
            }
        return {"model_calls": llamadas}

    @after_model(state_schema=HooksState)
    def post_model(state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        pedidas = [call["name"] for call in state["messages"][-1].tool_calls]
        if pedidas:
            print(f"[post_model] tools pedidas: {pedidas}")
            return {"tools_called": [*state.get("tools_called", []), *pedidas]}
        return None

    @after_agent(state_schema=HooksState)
    def post_agent(state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        duracion = time.time() - state.get("started_at", time.time())
        print(
            f"[post_agent] duración={duracion:.2f}s, llamadas al modelo={state.get('model_calls', 0)}, "
            f"tools usadas={state.get('tools_called', [])}"
        )
        return None

    return [pre_agent, pre_model, post_model, post_agent]


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=crear_hooks(max_model_calls=5),
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print("\n🤖", response["messages"][-1].content)

Ahora con un límite de **1** llamada. Esta pregunta necesita dos (una para pedir la tool y otra para la respuesta final), así que el pre-hook corta antes de la segunda y el agente termina con nuestro mensaje:

In [ ]:
agent_limitado = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=crear_hooks(max_model_calls=1),
)

response = agent_limitado.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print("\n🤖", response["messages"][-1].content)

Este es el mismo diseño que usa el API: `MAX_MODEL_CALLS` evita que un bucle de tools se descontrole. Es control de flujo básico, no un mecanismo de seguridad.

## 8. Middleware como clase: `AgentMiddleware`

Con decoradores, cada función es un middleware. Cuando varios hooks forman una unidad lógica, se agrupan en una **clase** que hereda de `AgentMiddleware`: los métodos se llaman igual que los hooks (`before_agent`, `before_model`, `after_model`, `after_agent`) y la clase puede guardar configuración en `__init__`. También permite definir versiones asíncronas (`abefore_model`, etc.).

In [ ]:
from langchain.agents.middleware import AgentMiddleware


class ObservabilidadMiddleware(AgentMiddleware):
    state_schema = HooksState

    def __init__(self, etiqueta: str = "obs"):
        super().__init__()
        self.etiqueta = etiqueta

    def before_agent(self, state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[{self.etiqueta}] before_agent")
        return {"model_calls": 0}

    def before_model(self, state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        llamadas = state.get("model_calls", 0) + 1
        print(f"[{self.etiqueta}] before_model #{llamadas}")
        return {"model_calls": llamadas}

    def after_model(self, state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[{self.etiqueta}] after_model")
        return None

    def after_agent(self, state: HooksState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"[{self.etiqueta}] after_agent (llamadas al modelo: {state.get('model_calls', 0)})")
        return None


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[ObservabilidadMiddleware("clase")],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print("\n🤖", response["messages"][-1].content)

## 9. Complemento: hooks *wrap-style*

Los hooks *wrap* no corren "antes" o "después": **envuelven** la llamada. Reciben un `handler` y tú decides si lo ejecutas, cuántas veces y qué haces alrededor. Sirven para medir tiempos, reintentar, cambiar el request o devolver una respuesta sin llamar al modelo.

- `@wrap_model_call` envuelve cada llamada al LLM.
- `@wrap_tool_call` envuelve cada ejecución de una tool.

In [ ]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call


@wrap_model_call
def medir_modelo(request, handler):
    inicio = time.time()
    response = handler(request)  # llama al modelo (o al siguiente middleware)
    print(f"[wrap_model] el modelo tardó {time.time() - inicio:.2f}s")
    return response


@wrap_tool_call
def registrar_tool(request, handler):
    print(f"[wrap_tool] ejecutando {request.tool_call['name']}({request.tool_call['args']})")
    resultado = handler(request)
    print(f"[wrap_tool] resultado: {resultado.content!r}")
    return resultado


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    middleware=[medir_modelo, registrar_tool],
)

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Lima?")]})
print("\n🤖", response["messages"][-1].content)

## 10. Orden de ejecución con varios middleware

Cuando pasas `middleware=[A, B]`:

- Los hooks **before** corren en orden: A → B.
- Los hooks **after** corren en orden **inverso**: B → A.
- Los hooks **wrap** se anidan como llamadas de función: A envuelve a B, y B envuelve a la llamada real.

Lo comprobamos con dos middleware que solo imprimen su nombre:

In [ ]:
class Etiquetador(AgentMiddleware):
    def __init__(self, nombre: str):
        super().__init__()
        self.nombre = nombre

    def before_model(self, state, runtime: Runtime):
        print(f"  before_model  → {self.nombre}")

    def after_model(self, state, runtime: Runtime):
        print(f"  after_model   → {self.nombre}")

    def wrap_model_call(self, request, handler):
        print(f"  wrap entra    → {self.nombre}")
        response = handler(request)
        print(f"  wrap sale     ← {self.nombre}")
        return response


# LangChain identifica cada middleware por el nombre de su clase y no acepta duplicados,
# por eso creamos dos subclases distintas.
class MiddlewareA(Etiquetador):
    pass


class MiddlewareB(Etiquetador):
    pass


agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM_PROMPT,
    middleware=[MiddlewareA("A"), MiddlewareB("B")],
)

response = agent.invoke({"messages": [HumanMessage(content="Dime un saludo corto.")]})

Deberías ver: `before_model` A, B → `wrap` entra A, B → *(modelo)* → `wrap` sale B, A → `after_model` **B, A**.

## Resumen

- Un **hook** es una función que LangChain ejecuta en un punto fijo del ciclo del agente; se registra con `middleware=[...]` en `create_agent`.
- **Node-style**: `before_agent` y `after_agent` corren una vez por invocación; `before_model` y `after_model` corren en cada llamada al LLM.
- **Wrap-style**: `wrap_model_call` y `wrap_tool_call` envuelven la llamada y controlan si se ejecuta.
- Los hooks comparten datos con un estado propio (`AgentState` + `state_schema`) y pueden cortar la ejecución con `jump_to`.
- Se pueden escribir con decoradores (un hook por función) o como clase `AgentMiddleware` (varios hooks y configuración).
- Orden con varios middleware: `before` en orden, `after` en reverso, `wrap` anidado.

**En el API** (`04_agent_hooks`), los cuatro hooks de las secciones 6 y 7 viven en `src/services/hooks.py` (con el límite configurable `MAX_MODEL_CALLS`) y se registran con `create_agent(..., middleware=HOOKS)` en `src/services/agent_service.py`. En la siguiente clase usaremos esta misma base para agregar guardrails de seguridad.